# Assignment 2

 **This was supoosed to be 2 assignment but is a big assignment So go slow and learn what you are doing have fun**


# Part 1: Data Ingestion

Data Ingestion is the first step in a RAG pipeline. It involves collecting, reading, and loading raw data from various sources (such as PDFs, HTML, or databases) into the system. Here, we read all PDF files in a given directory and parse their content into structured documents containing page content and metadata.


Here we Doing only for pdf but in final project we will do for pdf,csv,xlsx,docx,txt.
if you want you can practise to extract data from one more file format i would love to see you do this.

In [2]:
!pip install -q langchain-community
!pip install pypdf
!pip install chromadb
!pip uninstall -y chromadb opentelemetry-api opentelemetry-sdk
!pip install chromadb==0.5.5 opentelemetry-api==1.27.0 opentelemetry-sdk==1.27.0

Found existing installation: chromadb 0.5.5
Uninstalling chromadb-0.5.5:
  Successfully uninstalled chromadb-0.5.5
Found existing installation: opentelemetry-api 1.27.0
Uninstalling opentelemetry-api-1.27.0:
  Successfully uninstalled opentelemetry-api-1.27.0
Found existing installation: opentelemetry-sdk 1.27.0
Uninstalling opentelemetry-sdk-1.27.0:
  Successfully uninstalled opentelemetry-sdk-1.27.0
  Using cached chromadb-0.5.5-py3-none-any.whl.metadata (6.8 kB)
  Using cached opentelemetry_api-1.27.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_sdk-1.27.0-py3-none-any.whl.metadata (1.5 kB)
Using cached chromadb-0.5.5-py3-none-any.whl (584 kB)
Using cached opentelemetry_api-1.27.0-py3-none-any.whl (63 kB)
Using cached opentelemetry_sdk-1.27.0-py3-none-any.whl (110 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-g

In [3]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


/tmp/ipykernel_10010/2076300660.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [4]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    # TODO: Implement PDF ingestion using PyPDFLoader.
    # Load all pdf files recursively from pdf_directory (using Path(pdf_directory).glob('**/*.pdf')).
    # For each page document loaded, add metadata fields:
    # - 'source_file': the filename of the PDF file
    # - 'file_type': 'pdf'
    # Collect all loaded documents in a list and return them.
    # Hint: Use PyPDFLoader(str(pdf_file)).load()
    all_documents = []
    pdf_files = Path(pdf_directory).glob("**/*.pdf")

    for pdf_file in pdf_files:

        loader = PyPDFLoader(str(pdf_file))
        documents = loader.load()
        for doc in documents:
            doc.metadata["source_file"] = pdf_file.name
            doc.metadata["file_type"] = "pdf"

        all_documents.extend(documents)

    return all_documents


In [5]:
from google.colab import files

uploaded = files.upload()

Saving sample_rag_document.pdf to sample_rag_document (4).pdf


In [6]:
pdf_directory = "/content"

In [7]:
all_pdf_documents= process_all_pdfs(pdf_directory)

# Part 2: Chunking

Chunking is the process of breaking down large documents into smaller, cohesive segments (chunks). Since Large Language Models (LLMs) have a limited context window (input size limit) and retrieve information more accurately from smaller context blocks, we must split large documents. In this assignment, you need to use **RecursiveCharacterTextSplitter** to split loaded documents into smaller paragraphs with proper overlap to maintain context between boundary lines.


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    # TODO: Implement text splitting using RecursiveCharacterTextSplitter.
    # You need to use RecursiveCharacterTextSplitter with:
    # - chunk_size=chunk_size of your choice
    # - chunk_overlap=chunk_overlap (recommended 20% of chunk_size)
    # - length_function=len
    # - separators=['\n\n', '\n', ' ', '']
    # Print how many documents were split into how many chunks. (Tip: do not use too big files it will it time)
    # Return the split documents list.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    return split_docs
    pass


In [10]:
chunks=split_documents(all_pdf_documents)
chunks

Split 6 documents into 6 chunks.


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-11T10:54:06+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-11T10:54:06+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/content/sample_rag_document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sample_rag_document.pdf', 'file_type': 'pdf'}, page_content='Sample PDF for RAG Data Ingestion Assignment\nThis is a sample PDF created for testing PDF ingestion using PyPDFLoader or\nPyMuPDFLoader in a Retrieval-Augmented Generation (RAG) pipeline.\nThe quick brown fox jumps over the lazy dog. Artificial Intelligence helps process\ndocuments, create embeddings, and answer questions from retrieved context.'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-11T11:00:42+00:00', 'author': '(anonymous)', 'keyword

# Part 3: Embedding

Embedding is the process of converting text blocks into numerical vector representations. These vectors capture the semantic meaning of the text. Sentences that are semantically similar will be closer to each other in the vector space. We use pre-trained sentence transformer models (like 'all-MiniLM-L6-v2') to convert text chunks and queries into embeddings.

---



from sentence_transformers import SentenceTransformer

Imports the embedding model class.

This library:

downloads pretrained transformer models
converts text → embeddings

Built on top of:

HuggingFace transformers
PyTorch

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid #to get each chunk a unique id after embedding
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        # TODO: Load the SentenceTransformer model using self.model_name.
        # Print a message stating the model name is being loaded.
        # Set self.model to the loaded SentenceTransformer.
        # Print a message indicating successful loading and print the embedding dimension.
        # Print loading message

        print(f"Loading embedding model: {self.model_name}...")
        self.model = SentenceTransformer(self.model_name)
        print("Model loaded successfully!")
        print(f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        # TODO: Generate embeddings for the given list of texts using the model.
        # Make sure to handle cases where self.model is None.
        # Use self.model.encode(texts, show_progress_bar=True).
        # Return the resulting numpy array of embeddings.
        if self.model is None:
            raise ValueError("Embedding model is not loaded.")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        return embeddings




# Part 4: Vector DB

Vector Database (Vector DB) is a specialized database designed to store and query high-dimensional vector embeddings efficiently. It allows us to persist our document chunks along with their computed embeddings and perform fast search operations. Here, we use ChromaDB, which runs locally and stores documents persistently in a directory.

In [26]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        # TODO: Create the persist directory if it doesn't exist.
        # Initialize chromadb.PersistentClient with path=self.persist_directory.
        # Get or create a collection using self.client.get_or_create_collection
        # with self.collection_name and description metadata.
        # Create the persist directory if it doesn't exist
     # Create directory
        os.makedirs(self.persist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "PDF document embeddings"
            }
        )

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        # TODO: Add documents to ChromaDB collection.
        # 1. Check if the number of documents matches the number of embeddings.
        # 2. For each document, generate a unique ID using uuid.uuid4().hex[:8]
        # 3. Construct metadata dict including original document metadata, doc_index, and content_length.
        # 4. Extract document content string.
        # 5. Convert embedding to a list using .tolist().
        # 6. Add ids, embeddings, metadatas, and documents to self.collection.
        # Check lengths
        if len(documents) != len(embeddings):
         raise ValueError(
            "Number of documents and embeddings must match."
        )

        ids = []
        metadatas = []
        docs = []
        vectors = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

          unique_id = uuid.uuid4().hex[:8]
          ids.append(unique_id)

          metadata = dict(doc.metadata)
          metadata["doc_index"] = i
          metadata["content_length"] = len(doc.page_content)
          metadatas.append(metadata)

          docs.append(doc.page_content)

          vectors.append(embedding.tolist())

          self.collection.add(
             ids=ids,
            embeddings=vectors,
            metadatas=metadatas,
            documents=docs
          )

        print(f"Added {len(documents)} documents to the vector store.")



In [27]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-11T10:54:06+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-11T10:54:06+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/content/sample_rag_document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sample_rag_document.pdf', 'file_type': 'pdf'}, page_content='Sample PDF for RAG Data Ingestion Assignment\nThis is a sample PDF created for testing PDF ingestion using PyPDFLoader or\nPyMuPDFLoader in a Retrieval-Augmented Generation (RAG) pipeline.\nThe quick brown fox jumps over the lazy dog. Artificial Intelligence helps process\ndocuments, create embeddings, and answer questions from retrieved context.'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-11T11:00:42+00:00', 'author': '(anonymous)', 'keyword

## convert text to embeddings


In [28]:
### lets see text of chumks
texts=[doc.page_content for doc in chunks]

texts

['Sample PDF for RAG Data Ingestion Assignment\nThis is a sample PDF created for testing PDF ingestion using PyPDFLoader or\nPyMuPDFLoader in a Retrieval-Augmented Generation (RAG) pipeline.\nThe quick brown fox jumps over the lazy dog. Artificial Intelligence helps process\ndocuments, create embeddings, and answer questions from retrieved context.',
 'Sample PDF for Code Verification\nThis is a sample PDF created for testing PDF loading and text extraction pipelines.\nSection 1: Artificial Intelligence enables machines to perform tasks that typically\nrequire human intelligence.\nSection 2: Machine Learning is a subset of AI where systems learn patterns from data.\nSection 3: Retrieval-Augmented Generation (RAG) combines document retrieval with\nlarge language models.\nKeywords: PDF, testing, embeddings, vector database, chunking, retrieval, document\nprocessing.',
 'Sample PDF for RAG Data Ingestion Assignment\nThis is a sample PDF created for testing PDF ingestion using PyPDFLoader 

In [31]:
## Generate the Embeddings
embedding_manager = EmbeddingManager()

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore = VectorStore()
vectorstore.add_documents(chunks,embeddings)

Loading embedding model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!
Embedding dimension: 384


/tmp/ipykernel_10010/2856084859.py:26: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Added 6 documents to the vector store.


# Part 5: Query Retrieval

Query Retrieval starts with the user entering a natural language query. We must convert this query into the same embedding space using our embedding manager. This encoded query is then sent to the vector database to retrieve raw results.

---

# Part 6: Similarity Search

Similarity Search is the mathematical calculation (such as Cosine Similarity) used by the vector store to compare the query embedding with document embeddings. It ranks and returns the top_k most similar documents. We can filter results by a minimum similarity score (score_threshold) to keep only relevant context.


In [34]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        # TODO: Retrieve relevant documents from the ChromaDB collection.
        # 1. Generate query embedding using self.embedding_manager.generate_embeddings.
        # 2. Query the vector store collection with the query embedding list, asking for top_k results.
        # 3. For each result, convert distance to similarity score (similarity_score = 1 - distance).
        # 4. Filter results that have similarity_score >= score_threshold.
        # 5. Return a list of dicts with keys: 'id', 'content', 'metadata', 'similarity_score', 'distance', 'rank'.
        # 1. Generate query embedding
        # 1. Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])

        results = self.vector_store.collection.query(
          query_embeddings=query_embedding.tolist(),
          n_results=top_k
        )

        retrieved_docs = []

        for rank, (
          doc_id,
          document,
          metadata,
          distance,
        ) in enumerate(
          zip(
            results["ids"][0],
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
          ),
          start=1,
        ):

          similarity_score = 1 - distance

          if similarity_score >= score_threshold:
            retrieved_docs.append(
                {
                    "id": doc_id,
                    "content": document,
                    "metadata": metadata,
                    "similarity_score": similarity_score,
                    "distance": distance,
                    "rank": rank,
                }
            )

        return retrieved_docs


In [36]:
rag_retriever = RAGRetriever(
    vector_store=vectorstore,
    embedding_manager=embedding_manager
)

In [40]:
rag_retriever.retrieve(
    "quick brown fox",
    top_k=5,
    score_threshold=0
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[]

# Part 7: Prompting and Calling LLM

Prompting and Calling LLM is the synthesis step of RAG. We take the retrieved contexts, format them into a structured prompt alongside the user's original query, and pass them to the Large Language Model (LLM) to generate a grounded, factually accurate response.


In [41]:
def rag_simple(query,retriever,llm,top_k=3):
    # TODO: Implement a simple RAG pipeline using following steps which join above functions.
    # 1. Use the retriever to fetch top_k documents for the query.
    # 2. Join the content of the retrieved documents to form the context.
    # 3. Format a prompt instructing the LLM to use the context to answer the question.
    # 4. Call the LLM with the formatted prompt and return the string content of the response.
    """
    Simple RAG pipeline
    """

    retrieved_docs = retriever.retrieve(
        query=query,
        top_k=top_k
    )

    context = "\n\n".join(
        [doc["content"] for doc in retrieved_docs]
    )

    prompt = f"""
Use the following context to answer the user's question.
If the answer is not present in the context, say that you don't know.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content



In [ ]:
answer=rag_simple("three reasons for forgetting",rag_retriever,llm)
print(answer)

# Part 8: Advanced RAG (Optional)

Advanced RAG includes sophisticated pipeline elements such as streaming responses, citation tracking, interaction history (conversational memory), response summarization, and score-based filtering to make the application robust and production-ready.


In [ ]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    # TODO: Implement advanced RAG retrieval and generation.
    # 1. Retrieve top_k documents using score threshold min_score.
    # 2. Parse sources including source_file, page, similarity_score, and a content preview.
    # 3. Determine confidence (e.g. max similarity score of retrieved docs).
    # 4. Invoke LLM with formatted prompt combining context and query.
    # 5. Return dict containing 'answer', 'sources', 'confidence' (and 'context' if return_context is True).
    pass


In [ ]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # TODO: Implement the AdvancedRAGPipeline query logic:
        # 1. Retrieve documents using self.retriever.
        # 2. Parse sources/metadata and construct the context.
        # 3. If stream=True, simulate streaming by printing the prompt in chunks.
        # 4. Generate the answer using self.llm.
        # 5. Construct citations list and append it to the answer.
        # 6. If summarize=True, call the LLM to get a 2-sentence summary of the answer.
        # 7. Append the question, answer, sources, and summary to self.history.
        # 8. Return dictionary containing question, answer (with citations), sources, summary, and history.
        pass
